In [ ]:

import numpy as np
import os
import pandas as pd
from functools import reduce
import pickle
import myutils as uti
from sklearn.preprocessing import StandardScaler,Normalizer
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pandas as pd
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings("ignore")
from matplotlib import rcParams
plt.switch_backend('agg')
rcParams['font.weight'] = 'bold'



def read_pickle(modality_name):

    path='./all_data.csv'

    data_df = pd.read_csv(path) 
    
    
    
    
    all_quality=data_df.loc[:,"ppg_quality"].to_numpy()

    all_label=data_df.loc[:,"label"].to_numpy()
    all_user=data_df.loc[:,"user"].to_numpy()
    all_task=data_df.loc[:,"task"].to_numpy()
    
    all_data=data_df.loc[:, ~data_df.columns.isin(['ppg_quality', 'label',"user","task",'Unnamed: 0'])]#.to_numpy()
    
    
    #m1 = np.quantile(all_quality, 0.95, axis=0)
    #m2 = np.quantile(all_quality, 0.05, axis=0)
    
    #plt.hist(all_quality)
    #plt.savefig('./quality.png')
    #plt.cla()
    #plt.clf()
    
    
    
    drop_list=['HRV_SDANN1','HRV_SDNNI1','HRV_SDANN2','HRV_SDNNI2','HRV_SDANN5','HRV_SDNNI5','HRV_VLF','HRV_ULF','Phasic_phasic_entropy','BVP_BVP_entropy']
    all_data=all_data.drop(drop_list, axis=1)
    
    all_data.replace([np.inf, -np.inf], np.nan, inplace=True)
     
    all_data=all_data.fillna(0)
   
    #nan_count = all_data.isin([np.inf, -np.inf]).sum()
    
    #nan_count.to_csv('nan_count.csv') 
    #"all",'eda','edabvp','edahr','edahrv','edatemp','edaacc'
    if modality_name=='all':
        
        modality=['EDA','HRV','Phasic','Tonic','TEMP','HR','ACC','BVP']
    elif modality_name=='eda':
        modality=['EDA','Phasic','Tonic']
    elif modality_name=='edabvp':
        modality=['EDA','Phasic','Tonic','BVP']
    elif modality_name=='edahr':
        modality=['EDA','Phasic','Tonic','HR']
    elif modality_name=='edahrv':
        modality=['EDA','Phasic','Tonic','HRV']
    elif modality_name=='edatemp':
        modality=['EDA','Phasic','Tonic','TEMP']
    elif modality_name=='edaacc':
        modality=['EDA','Phasic','Tonic','ACC']
  
    
    names=all_data.columns
    keep=[]

    for name in names:
        moda=name.split("_")[0]
        if moda in modality:
            keep.append(name)
    

    all_data=all_data.loc[:,all_data.columns.isin(keep)]


            
    all_data=all_data.to_numpy()

    
    



    
    return all_data,all_label,all_user,all_task,all_quality
    
    
    
def get_data(  modality_name='all',data_name=0,fold_idx=0,irm_batch=False,batch_size=50):
    
    label_dict={'calm':0,'stress':1}
    task_set=['jelly','count','stress','prepareSong','arithmetic','bad','calm','cycling1','cycling2','run1','run2','sit','stand','baseline','stroop','HAHV_sit','LAHV_sit','LALV_sit','HALV_sit','HAHV_walk','LAHV_walk','LALV_walk','HALV_walk','City1_Start','Rest_Start','Hwy_Start', 'City2_Start', 'City2_Start.1', 'Hwy_Start.1','City1_Start.1',  ]#'animal','good''Rest_Start.1',
    
    #path='/home/yxiao54/physiology/data/windows/window_data.pickle'
    all_datas,all_label,all_user,all_task,all_quality=read_pickle(modality_name)

    control=[   "0001","0002","0003","0004","0005","0006","0007","0008","0009","0010","0011","0012","0013","0014","0015","0016","xianfei",'kirat', 'lohith7', 'lorn', 'lydia7',
 'missy', 'moira2', 'ruipeng', 'sawin', 'shaily5', 'shocky', 'susan8', 'tabitha5','timothy', 'vicki', 'xianfei', 'yuhsun', 'yuxuan',  'gabrielle', 'coung','georgie', 'hannah','avery' ,'brian', 'carolyn','christine','huaiyu', 'jingyu','jordan', 'joshua', 'kaite', 'kara8']
       
    oud=["8803","8804","8814","8829","8832","8840","8847","8852","8876","8898","9913","9915","9915v2","9925","9929","9933","9933v2","9941","9945","9945v2","9948","9952","9956","9967","9969","9973","9973v2","9984","9991"]
    
    predose=['8803','8804','8814','8829','8832','8840','8847','8876','8898','9913','9915v2','9933v2','9945v2','9973v2']
    postdose=['8852','9915','9925','9929','9933','9941','9945','9948','9952','9956','9967','9969','9973','9984','9991']

       
    exercise=["P01","P02","P03","P04","P05","P06","P07","P08","P09","P10","P11","P12","P13","P15","P16","P17"]
    wesad=['S10' ,'S11','S13' ,'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']
    
    alcohole=["Part101C","Part102C","Part104C","Part105C","Part106C","Part107C","Part108C","Part109C","Part110C","Part111C","Part112C"]
       
    control_v2=[ "alenav2","anjaliv2","brianav2","domv2","fabricv2","hanav2","jamiev2","karav2","kristinav2","nanv2","rubyv2","tomv2","xinhaov2"]
    
    
    emo=['1-E4-Drv1' ,'10-E4-Drv10',
 '11-E4-Drv11' ,'12-E4-Drv12' ,'13-E4-Drv13' ,'3-E4-Drv3' ,'4-E4-Drv4',
 '5-E4-Drv5' ,'6-E4-Drv6', '7-E4-Drv7' ,'8-E4-Drv8']
 
    
    
    
    test_control=['0006' ,'jordan', 'vicki','susan8','shaily5','0008','carolyn','0002','0016','avery','brian','0007','missy','0003']
    
    control=['user_'+x for x in control]
    
    predose=['user_'+x for x in predose]
    postdose=['user_'+x for x in postdose]
    wesad=['user_'+x for x in wesad]
    emo=['user_'+x for x in emo]
    
    
    train_users=control


    task_idx=np.argwhere(np.isin(all_task, task_set)).ravel()

    
    
    quality_idx=np.argwhere(all_quality<20)



      
    loader_list=[]
    user_list=[train_users,predose,postdose,wesad,emo]
    
    name_list=['control','predose','postdose','wesad','affetiveroad']
    
    
    for name,user in zip(name_list,user_list):
        user_idx=np.argwhere(np.isin(all_user, user)).ravel()
        select_idx=reduce(np.intersect1d, [user_idx, task_idx, quality_idx])
    
        test_label=all_label[select_idx]
        #test_label=list(map(lambda x: label_dict[x], test_label))
        test_user=all_user[select_idx]
        test_task=all_task[select_idx] 
    
        test_data=all_datas[select_idx]
        
        
        #test_data,test_user,test_label,test_task=uti.user_changeScore(test_data,test_user,test_label,test_task)
        test_data,test_user,test_label,test_task=uti.user_changeScore2(test_data,test_user,test_label,test_task)
        
        loader_list.append((name,test_data,test_user,test_label,test_task))
        
    return loader_list
all_names=['num_onset','num_peaks','num_recovery','height','amplitude','risetime','recoverytime','BVP_BVP_mean','BVP_BVP_std','BVP_BVP_min','BVP_BVP_max','BVP_BVP_ptp',
    'BVP_BVP_sum','BVP_BVP_energy','BVP_BVP_skewness','BVP_BVP_kurtosis','BVP_BVP_peaks','BVP_BVP_rms','BVP_BVP_lineintegral','BVP_BVP_n_above_mean','BVP_BVP_n_below_mean',
    'BVP_BVP_n_sign_changes','BVP_BVP_iqr','BVP_BVP_iqr_5_95','BVP_BVP_pct_5','BVP_BVP_pct_95','BVP_BVP_entropy','BVP_BVP_perm_entropy','BVP_BVP_svd_entropy','BVP_l2_mean',
    'BVP_l2_std','BVP_l2_min','BVP_l2_max','BVP_l2_ptp','BVP_l2_sum','BVP_l2_energy','BVP_l2_skewness','BVP_l2_kurtosis','BVP_l2_peaks','BVP_l2_rms','BVP_l2_lineintegral',
    'BVP_l2_n_above_mean','BVP_l2_n_below_mean','BVP_l2_n_sign_changes','BVP_l2_iqr','BVP_l2_iqr_5_95','BVP_l2_pct_5','BVP_l2_pct_95','BVP_l2_entropy','BVP_l2_perm_entropy',
    'BVP_l2_svd_entropy','HR_HR_mean','HR_HR_std','HR_HR_min','HR_HR_max','HR_HR_ptp','HR_HR_sum','HR_HR_energy','HR_HR_skewness','HR_HR_kurtosis','HR_HR_peaks','HR_HR_rms',
    'HR_HR_lineintegral','HR_HR_n_above_mean','HR_HR_n_below_mean','HR_HR_n_sign_changes','HR_HR_iqr','HR_HR_iqr_5_95','HR_HR_pct_5','HR_HR_pct_95','HR_HR_entropy','HR_HR_perm_entropy',
    'HR_HR_svd_entropy','HR_l2_mean','HR_l2_std','HR_l2_min','HR_l2_max','HR_l2_ptp','HR_l2_sum','HR_l2_energy','HR_l2_skewness','HR_l2_kurtosis','HR_l2_peaks','HR_l2_rms',
    'HR_l2_lineintegral','HR_l2_n_above_mean','HR_l2_n_below_mean','HR_l2_n_sign_changes','HR_l2_iqr','HR_l2_iqr_5_95','HR_l2_pct_5','HR_l2_pct_95','HR_l2_entropy','HR_l2_perm_entropy'
    ,'HR_l2_svd_entropy','TEMP_TEMP_mean','TEMP_TEMP_std','TEMP_TEMP_min','TEMP_TEMP_max','TEMP_TEMP_ptp','TEMP_TEMP_sum','TEMP_TEMP_energy','TEMP_TEMP_skewness','TEMP_TEMP_kurtosis',
    'TEMP_TEMP_peaks','TEMP_TEMP_rms','TEMP_TEMP_lineintegral','TEMP_TEMP_n_above_mean','TEMP_TEMP_n_below_mean','TEMP_TEMP_n_sign_changes','TEMP_TEMP_iqr','TEMP_TEMP_iqr_5_95',
    'TEMP_TEMP_pct_5','TEMP_TEMP_pct_95','TEMP_TEMP_entropy','TEMP_TEMP_perm_entropy','TEMP_TEMP_svd_entropy','TEMP_l2_mean','TEMP_l2_std','TEMP_l2_min','TEMP_l2_max','TEMP_l2_ptp',
    'TEMP_l2_sum','TEMP_l2_energy','TEMP_l2_skewness','TEMP_l2_kurtosis','TEMP_l2_peaks','TEMP_l2_rms','TEMP_l2_lineintegral','TEMP_l2_n_above_mean','TEMP_l2_n_below_mean',
    'TEMP_l2_n_sign_changes','TEMP_l2_iqr','TEMP_l2_iqr_5_95','TEMP_l2_pct_5','TEMP_l2_pct_95','TEMP_l2_entropy','TEMP_l2_perm_entropy','TEMP_l2_svd_entropy','ACC_x_mean',
    'ACC_x_std','ACC_x_min','ACC_x_max','ACC_x_ptp','ACC_x_sum','ACC_x_energy','ACC_x_skewness','ACC_x_kurtosis','ACC_x_peaks','ACC_x_rms','ACC_x_lineintegral','ACC_x_n_above_mean',
    'ACC_x_n_below_mean','ACC_x_n_sign_changes','ACC_x_iqr','ACC_x_iqr_5_95','ACC_x_pct_5','ACC_x_pct_95','ACC_x_entropy','ACC_x_perm_entropy','ACC_x_svd_entropy','ACC_y_mean','ACC_y_std','ACC_y_min','ACC_y_max','ACC_y_ptp','ACC_y_sum','ACC_y_energy','ACC_y_skewness','ACC_y_kurtosis','ACC_y_peaks','ACC_y_rms','ACC_y_lineintegral','ACC_y_n_above_mean','ACC_y_n_below_mean','ACC_y_n_sign_changes','ACC_y_iqr','ACC_y_iqr_5_95','ACC_y_pct_5','ACC_y_pct_95','ACC_y_entropy','ACC_y_perm_entropy','ACC_y_svd_entropy','ACC_z_mean','ACC_z_std','ACC_z_min','ACC_z_max','ACC_z_ptp','ACC_z_sum','ACC_z_energy','ACC_z_skewness','ACC_z_kurtosis','ACC_z_peaks','ACC_z_rms','ACC_z_lineintegral','ACC_z_n_above_mean','ACC_z_n_below_mean','ACC_z_n_sign_changes','ACC_z_iqr','ACC_z_iqr_5_95','ACC_z_pct_5','ACC_z_pct_95','ACC_z_entropy','ACC_z_perm_entropy','ACC_z_svd_entropy','ACC_l2_mean','ACC_l2_std','ACC_l2_min','ACC_l2_max','ACC_l2_ptp','ACC_l2_sum','ACC_l2_energy','ACC_l2_skewness','ACC_l2_kurtosis','ACC_l2_peaks','ACC_l2_rms','ACC_l2_lineintegral','ACC_l2_n_above_mean','ACC_l2_n_below_mean','ACC_l2_n_sign_changes','ACC_l2_iqr','ACC_l2_iqr_5_95','ACC_l2_pct_5','ACC_l2_pct_95','ACC_l2_entropy','ACC_l2_perm_entropy','ACC_l2_svd_entropy','HRV_MeanNN','HRV_SDNN','HRV_SDANN1','HRV_SDNNI1','HRV_SDANN2','HRV_SDNNI2','HRV_SDANN5','HRV_SDNNI5','HRV_RMSSD','HRV_SDSD','HRV_CVNN','HRV_CVSD','HRV_MedianNN','HRV_MadNN','HRV_MCVNN','HRV_IQRNN','HRV_SDRMSSD','HRV_Prc20NN','HRV_Prc80NN','HRV_pNN50','HRV_pNN20','HRV_MinNN','HRV_MaxNN','HRV_HTI','HRV_TINN','HRV_ULF','HRV_VLF','HRV_LF','HRV_HF','HRV_VHF','HRV_TP','HRV_LFHF','HRV_LFn','HRV_HFn','HRV_LnHF','Tonic_tonic_mean','Tonic_tonic_std','Tonic_tonic_min','Tonic_tonic_max','Tonic_tonic_ptp','Tonic_tonic_sum','Tonic_tonic_energy','Tonic_tonic_skewness','Tonic_tonic_kurtosis','Tonic_tonic_peaks','Tonic_tonic_rms','Tonic_tonic_lineintegral','Tonic_tonic_n_above_mean','Tonic_tonic_n_below_mean','Tonic_tonic_n_sign_changes','Tonic_tonic_iqr','Tonic_tonic_iqr_5_95','Tonic_tonic_pct_5','Tonic_tonic_pct_95','Tonic_tonic_entropy','Tonic_tonic_perm_entropy','Tonic_tonic_svd_entropy','Tonic_l2_mean','Tonic_l2_std','Tonic_l2_min','Tonic_l2_max','Tonic_l2_ptp','Tonic_l2_sum','Tonic_l2_energy','Tonic_l2_skewness','Tonic_l2_kurtosis','Tonic_l2_peaks','Tonic_l2_rms','Tonic_l2_lineintegral','Tonic_l2_n_above_mean','Tonic_l2_n_below_mean','Tonic_l2_n_sign_changes','Tonic_l2_iqr','Tonic_l2_iqr_5_95','Tonic_l2_pct_5','Tonic_l2_pct_95','Tonic_l2_entropy','Tonic_l2_perm_entropy','Tonic_l2_svd_entropy','Phasic_phasic_mean','Phasic_phasic_std','Phasic_phasic_min','Phasic_phasic_max','Phasic_phasic_ptp','Phasic_phasic_sum','Phasic_phasic_energy','Phasic_phasic_skewness','Phasic_phasic_kurtosis','Phasic_phasic_peaks','Phasic_phasic_rms','Phasic_phasic_lineintegral','Phasic_phasic_n_above_mean','Phasic_phasic_n_below_mean','Phasic_phasic_n_sign_changes','Phasic_phasic_iqr','Phasic_phasic_iqr_5_95','Phasic_phasic_pct_5','Phasic_phasic_pct_95','Phasic_phasic_entropy','Phasic_phasic_perm_entropy','Phasic_phasic_svd_entropy','Phasic_l2_mean','Phasic_l2_std','Phasic_l2_min','Phasic_l2_max','Phasic_l2_ptp','Phasic_l2_sum','Phasic_l2_energy','Phasic_l2_skewness','Phasic_l2_kurtosis','Phasic_l2_peaks','Phasic_l2_rms','Phasic_l2_lineintegral','Phasic_l2_n_above_mean','Phasic_l2_n_below_mean','Phasic_l2_n_sign_changes','Phasic_l2_iqr','Phasic_l2_iqr_5_95','Phasic_l2_pct_5','Phasic_l2_pct_95','Phasic_l2_entropy','Phasic_l2_perm_entropy','Phasic_l2_svd_entropy']
    
if __name__ == '__main__':
    
    
    data_list=get_data()
    df_list = []
    drop_list = ['HRV_SDANN1', 'HRV_SDNNI1', 'HRV_SDANN2', 'HRV_SDNNI2', 'HRV_SDANN5', 'HRV_SDNNI5', 'HRV_VLF', 'HRV_ULF', 'Phasic_phasic_entropy', 'BVP_BVP_entropy']
    features= [item for item in all_names if item not in drop_list]
    
    print(len(features))
    

340


In [6]:
for group_label, data, user_ids, task_markers, _ in data_list:
    df_temp = pd.DataFrame(data, columns=features)
    df_temp['user'] = user_ids                 # for inter-user variability
    df_temp['task_marker'] = task_markers        # for intra-user dependency (calm vs. stress)
    df_temp['Group_label'] = group_label         # the group you are in (control, predose, postdose)
    df_list.append(df_temp)

df_all = pd.concat(df_list, ignore_index=True)


In [7]:
groups_to_compare =[['control','predose'],['control','postdose'],['predose','postdose'],['wesad', 'affetiveroad'], ['control', 'wesad'], ['control', 'affetiveroad']]
for compare_groups in groups_to_compare:
    print(f"Comparing groups: {compare_groups[0]} (ref) vs {compare_groups[1]}")
    df=df_all[df_all['Group_label'].isin(compare_groups)]
    df['task_marker'] = pd.Categorical(df['task_marker'], categories=['calm', 'stress'], ordered=True)
    df['Group_label'] = pd.Categorical(df['Group_label'], categories=compare_groups, ordered=True)
    task_conditions = ['calm', 'stress']


    total_features = 0     
    sig_calm_count = 0
    sig_stress_count = 0
    sig_feature_summaries = []      
    nonconverged_features = {}      
    sig_threshold = 0.05
    effect_sizes = {'calm': {}, 'stress': {}}
    sig_features_calm = []
    sig_features_stress = []


    for feat in features:
        converged_for_feat = True  # flag whether both task condition models converge for this feature
        for task in task_conditions:
            df_task = df[df['task_marker'] == task]
            formula = f"{feat} ~ C(Group_label)"
            try:
                model = smf.mixedlm(formula, df_task, groups=df_task['user'])
                result = model.fit()
                print(result.summary())
                if not result.converged:
                    converged_for_feat = False
                    nonconverged_features[(feat, task)] = "Model did not converge."
                    print(f"Model for feature '{feat}' under task '{task}' did NOT converge.")
                    continue  # Skip significance check for this condition
            except Exception as err:
                converged_for_feat = False
                nonconverged_features[(feat, task)] = f"Error: {err}"
                print(f"Error fitting model for feature '{feat}' under task '{task}': {err}")
                continue

            
            if task == task_conditions[-1] and converged_for_feat:
                total_features += 1
            
            coef_key = f"C(Group_label)[T.{compare_groups[1]}]"
            group_coef_p = result.pvalues.get(coef_key, np.nan)
            group_coef = result.params.get(coef_key, np.nan)

            if task == 'calm':
                if group_coef_p < sig_threshold:
                    sig_calm_count += 1
                    sig_features_calm.append(feat)
            else:  # task == 'stress'
                if group_coef_p < sig_threshold:
                    sig_features_stress.append(feat)
                    sig_stress_count += 1

            # If significant in either condition, store a summary
            if group_coef_p < sig_threshold:
                summary_str = result.summary().as_text()
                sig_feature_summaries.append(
                    f"Feature: {feat} ({task} condition)\n"
                    f"p-value for Group ({compare_groups[1]} vs {compare_groups[0]}): {group_coef_p:.4f}\n"
                    f"{'-'*80}\n"
                    f"{summary_str}\n\n"
                )
                print(f"Feature '{feat}' under task '{task}' significant (p={group_coef_p:.4f}).")

    common_sig_features = list(set(sig_features_calm).intersection(set(sig_features_stress)))
    common_sig_count = len(common_sig_features)
    output_filename = f"./results/mixedlm_comparison_results_{compare_groups[0]}_{compare_groups[1]}.txt"
    with open(output_filename, "w") as f_out:
        f_out.write(f"Groups compared: {compare_groups[0]} (ref), {compare_groups[1]}\n")
        f_out.write(f"Total features analyzed (both task conditions converged): {total_features}\n")
        f_out.write(f"Features significant in calm condition: {sig_calm_count}\n")
        f_out.write(f"Features significant in stress condition: {sig_stress_count}\n")
        f_out.write(f"Features significant in both calm and stress conditions: {common_sig_count}\n")
        if common_sig_count > 0:
            f_out.write("Common significant features:\n")
            for feat in common_sig_features:
                f_out.write(f"  - {feat}\n")
        else:
            f_out.write("No features were significant in both conditions.\n")
        f_out.write("\nResult summary for significant features:\n")
        f_out.write("="*80 + "\n")
        for summary in sig_feature_summaries:
            f_out.write(summary)
        if nonconverged_features:
            f_out.write("\nWARNING: The following feature-task combinations did not converge:\n")
            for (feat, task), msg in nonconverged_features.items():
                f_out.write(f"Feature: {feat} | Task: {task} -> {msg}\n")
            
    print(f"Results written to {output_filename}")

Comparing groups: control (ref) vs predose
               Mixed Linear Model Regression Results
Model:                MixedLM     Dependent Variable:     num_onset 
No. Observations:     771         Method:                 REML      
No. Groups:           61          Scale:                  47.4913   
Min. group size:      6           Log-Likelihood:         -2606.0615
Max. group size:      25          Converged:              Yes       
Mean group size:      12.6                                          
--------------------------------------------------------------------
                          Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------------------
Intercept                  0.603    0.437  1.380 0.168 -0.254  1.461
C(Group_label)[T.predose] -0.395    0.898 -0.440 0.660 -2.154  1.365
Group Var                  5.011    0.277                           

              Mixed Linear Model Regression Results
Model:                M